# Fine-tuning DistilBERT for Humor: 128 tokens (Colab)

Trains DistilBERT to predict the rJokes 0–11 humor score, on Kaggle's GPU. Beat the local TF-IDF baseline of **Spearman ≈ 0.363**.


**Before you run:**
1. **Settings → Accelerator → GPU** (T4 x2 or P100).
2. **Settings → Internet → On**
3. **Add your Hugging Face token as a Secret:** Add-ons → Secrets → add one named
   `HF_TOKEN` with your HF **write** token.


In [ ]:
# CONFIG
GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/humor-intelligence.git"
HF_MODEL_REPO   = "YOUR_HF_USERNAME/humor-intelligence-distilbert"  # the 128 repo

MODEL_NAME  = "distilbert-base-uncased"
MAX_LENGTH  = 128
EPOCHS      = 2
BATCH_SIZE  = 32
LR          = 2e-5
SEED        = 42
CKPT_DIR    = "/kaggle/working/humor-distilbert-128"

In [ ]:
# 1. GPU check
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (!)")

In [ ]:
# 2. Install the HF stack
!pip install -q -U "transformers>=4.40" "datasets>=2.19" "accelerate>=0.30" scipy

In [ ]:
# 3. Authenticate to Hugging Face via the Kaggle Secret
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login
    login(token=hf_token)
    print("Logged in to Hugging Face via Kaggle secret.")
except Exception as e:
    print("No HF_TOKEN secret found — using interactive login. (", e, ")")
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
# 4. Clone repo & rebuild cleaned data
import os, shutil
if os.path.exists("humor-intelligence"):
    shutil.rmtree("humor-intelligence")
!git clone -q $GITHUB_REPO_URL
%cd humor-intelligence
!python src/data.py
%cd /kaggle/working

In [ ]:
# 5. Load cleaned splits
import pandas as pd
BASE = "humor-intelligence/data/processed"
train = pd.read_csv(f"{BASE}/train.csv")
dev   = pd.read_csv(f"{BASE}/dev.csv")
test  = pd.read_csv(f"{BASE}/test.csv")
print(f"train {len(train):,} | dev {len(dev):,} | test {len(test):,}")

In [ ]:
# 6. Tokenize at 128.
from datasets import Dataset, Value
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_ds(df):
    ds = Dataset.from_pandas(
        df[["joke", "score"]].rename(columns={"score": "labels"}),
        preserve_index=False,
    )
    ds = ds.map(lambda b: tokenizer(b["joke"], truncation=True,
                                    max_length=MAX_LENGTH), batched=True)
    ds = ds.cast_column("labels", Value("float32"))
    return ds

train_ds, dev_ds, test_ds = make_ds(train), make_ds(dev), make_ds(test)
print("label dtype:", train_ds.features["labels"].dtype)

In [ ]:
# 7. Regression model & Spearman metric
import numpy as np
from scipy.stats import spearmanr, pearsonr
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=1, problem_type="regression")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.asarray(preds).squeeze()
    return {"spearman": float(spearmanr(labels, preds).correlation),
            "pearson":  float(pearsonr(labels, preds)[0])}

In [ ]:
# 8. Train
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
from transformers.trainer_utils import get_last_checkpoint

collator = DataCollatorWithPadding(tokenizer)
args = TrainingArguments(
    output_dir=CKPT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    num_train_epochs=EPOCHS, learning_rate=LR, seed=SEED, fp16=True,
    eval_strategy="steps", eval_steps=500,
    save_strategy="steps", save_steps=500, save_total_limit=2,
    logging_steps=200,
    load_best_model_at_end=True, metric_for_best_model="spearman",
    greater_is_better=True, report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  eval_dataset=dev_ds, compute_metrics=compute_metrics,
                  data_collator=collator)
last = get_last_checkpoint(CKPT_DIR) if os.path.isdir(CKPT_DIR) else None
if last: print("Resuming from", last)
trainer.train(resume_from_checkpoint=last)

In [ ]:
# 9. Test-set result
test_metrics = trainer.evaluate(test_ds)
print("128-token TEST Spearman:", round(test_metrics["eval_spearman"], 4))
print("128-token TEST Pearson :", round(test_metrics["eval_pearson"], 4))
print("\nBaseline was 0.363.")

In [ ]:
# 10. Push to the HF repo (already authenticated in step 3)
trainer.push_to_hub(HF_MODEL_REPO)
tokenizer.push_to_hub(HF_MODEL_REPO)
print("Pushed to https://huggingface.co/" + HF_MODEL_REPO)